# 🔧 LoRA Fine-Tuning

## What is LoRA (Low-Rank Adaptation)?

LoRA is a technique for efficiently fine-tuning large language models (LLMs).

### Core Principle
- Adds a low-rank decomposition matrix **ΔW = BA** to the pre-trained weight matrix **W**
- Freezes the original weights W and only trains A and B
- Achieves strong performance while training **less than 1%** of total parameters

### Why LoRA?
| Advantage | Description |
|------|------|
| **Memory efficient** | Uses far less GPU memory compared to full model training |
| **Fast training** | Fewer trainable parameters lead to faster convergence |
| **Modularity** | Swap adapters to apply to different domains |
| **Preserves original** | Does not modify the base model weights |

### Configuration in this notebook
- **Base model**: `Qwen/Qwen3-4B-Instruct-2507`
- **LoRA rank (r)**: 16, **alpha**: 32
- **Target modules**: attention (Q/K/V/O) + MLP (gate/up/down)
- **Loss masking**: Only trains on assistant responses (system/user messages excluded)

> ⚠️ This notebook uses a **prepared data bundle**.  
> No SDG, teacher model calls, or data modification is performed.  
> Training will stop if the bundle is invalid.

In [ ]:
"""Load configuration and validate bundle compatibility."""

import os
from pathlib import Path

from rhoai_model_training_lab.config import (
    load_env, load_training_config, load_bundle_config, PROJECT_ROOT,
)
from rhoai_model_training_lab.data import BundleManager

load_env()

# Load LoRA config
lora_config = load_training_config("lora")
model_id = lora_config["model"]["model_id"]
model_revision = lora_config["model"]["model_revision"]

print(f"Model: {model_id} (rev: {model_revision})")
print(f"LoRA r={lora_config['lora']['r']}, alpha={lora_config['lora']['lora_alpha']}")
print(f"Target modules: {lora_config['lora']['target_modules']}")
print(f"Seed: {lora_config['training']['seed']}")

# Load and validate bundle
release_config = load_bundle_config()
bundle_base = release_config.get("bundle", {}).get("base_path", "data/prepared/tau-knowledge-v1")
bundle_path = PROJECT_ROOT / bundle_base

print(f"\nBundle path: {bundle_path}")
mgr = BundleManager.load_bundle(bundle_path)
manifest = mgr.manifest

# Compatibility check
compat = mgr.validate_compatibility(model_id)
if compat.errors:
    for err in compat.errors:
        print(f"  ❌ {err}")
    raise RuntimeError(
        "Bundle compatibility check failed — cannot proceed with training.\n"
        "Generate the correct bundle from the data_preparation/ notebooks."
    )

print(f"\n✅ Bundle compatibility check passed")
print(f"   Training samples: {manifest.canonical_train_count}")
print(f"   Validation samples: {manifest.canonical_validation_count}")
print(f"   Bundle version: {manifest.bundle_version}")

In [ ]:
"""Preview training data with loss mask visualization."""

from transformers import AutoTokenizer

# Load tokenizer for visualization
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# Load backend-specific training data
train_data = mgr.get_training_samples("lora", "train")
val_data = mgr.get_training_samples("lora", "validation")

print(f"LoRA training data: {len(train_data)} samples")
print(f"LoRA validation data: {len(val_data)} samples")

# Preview first 2 examples with loss mask visualization
print("\n" + "=" * 70)
print("📝 Training data preview (loss mask visualization)")
print("=" * 70)

for i, sample in enumerate(train_data[:2]):
    messages = sample.get("messages", [])
    print(f"\n--- Sample {i+1} ---")
    print(f"Messages: {len(messages)}")

    for msg in messages:
        role = msg["role"]
        content = msg.get("content", "") or ""
        # Loss mask: only assistant messages contribute to loss
        is_target = role == "assistant"
        mask_icon = "🎯 [training target]" if is_target else "🚫 [masked]"

        display_content = content[:200] + "..." if len(content) > 200 else content
        print(f"\n  {mask_icon} [{role}]:")
        print(f"    {display_content}")

        if msg.get("tool_calls"):
            print(f"    🔧 Tool calls: {len(msg['tool_calls'])}")
            for tc in msg["tool_calls"][:2]:
                fn = tc.get("function", {})
                print(f"       → {fn.get('name', '?')}({fn.get('arguments', '')[:80]})")

    # Token count
    try:
        formatted = tokenizer.apply_chat_template(messages, tokenize=True)
        print(f"\n  Token count: {len(formatted)}")
        max_len = lora_config["data"]["max_seq_length"]
        if len(formatted) > max_len:
            print(f"  ⚠️  Exceeds max length {max_len}!")
    except Exception:
        pass

print(f"\n{'=' * 70}")
print("Loss masking policy: only assistant responses are training targets")
print("System messages, user inputs, and tool observations are excluded from loss computation.")

In [ ]:
"""Train using training_hub.lora_sft — actual API call."""

import time
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not detected.\n"
        "LoRA training requires a CUDA-compatible GPU.\n"
        "Run this notebook in an environment with a GPU."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / (1024**3):.1f} GB")
print()

# Prepare training_hub arguments from config
train_file = str(PROJECT_ROOT / lora_config["data"]["train_file"])
val_file = str(PROJECT_ROOT / lora_config["data"]["validation_file"])
output_dir = str(PROJECT_ROOT / lora_config["training_args"]["output_dir"])

print("=" * 70)
print("🚀 Starting LoRA Training")
print("=" * 70)
print(f"  Training data: {train_file}")
print(f"  Validation data: {val_file}")
print(f"  Output directory: {output_dir}")
print(f"  Epochs: {lora_config['training_args']['num_train_epochs']}")
print(f"  Batch size: {lora_config['training_args']['per_device_train_batch_size']}")
print(f"  Gradient accumulation: {lora_config['training_args']['gradient_accumulation_steps']}")
print(f"  Learning rate: {lora_config['training_args']['learning_rate']}")
print()

start_time = time.time()

# Actual training_hub API call
from training_hub import lora_sft

training_result = lora_sft(
    model_id=model_id,
    model_revision=model_revision,
    train_file=train_file,
    validation_file=val_file,
    output_dir=output_dir,
    lora_r=lora_config["lora"]["r"],
    lora_alpha=lora_config["lora"]["lora_alpha"],
    lora_dropout=lora_config["lora"]["lora_dropout"],
    target_modules=lora_config["lora"]["target_modules"],
    num_train_epochs=lora_config["training_args"]["num_train_epochs"],
    per_device_train_batch_size=lora_config["training_args"]["per_device_train_batch_size"],
    gradient_accumulation_steps=lora_config["training_args"]["gradient_accumulation_steps"],
    learning_rate=lora_config["training_args"]["learning_rate"],
    weight_decay=lora_config["training_args"]["weight_decay"],
    warmup_ratio=lora_config["training_args"]["warmup_ratio"],
    lr_scheduler_type=lora_config["training_args"]["lr_scheduler_type"],
    max_seq_length=lora_config["data"]["max_seq_length"],
    bf16=lora_config["training_args"]["bf16"],
    gradient_checkpointing=lora_config["training_args"]["gradient_checkpointing"],
    seed=lora_config["training"]["seed"],
    logging_steps=lora_config["training_args"]["logging_steps"],
    eval_steps=lora_config["training_args"]["eval_steps"],
    save_steps=lora_config["training_args"]["save_steps"],
    save_total_limit=lora_config["training_args"]["save_total_limit"],
    resume_from_checkpoint=lora_config["training_args"].get("resume_from_checkpoint", True),
)

wall_time = time.time() - start_time

print(f"\n✅ LoRA training complete!")
print(f"  Wall time: {wall_time/60:.1f} min")
print(f"  Final train loss: {getattr(training_result, 'train_loss', 'N/A')}")
print(f"  Final eval loss: {getattr(training_result, 'eval_loss', 'N/A')}")

In [ ]:
"""Inspect training results — loss curve and metrics."""

import json

# Load training logs
log_history = getattr(training_result, "log_history", None)
output_dir_path = Path(output_dir)

if log_history is None:
    # Try loading from trainer_state.json
    state_path = output_dir_path / "trainer_state.json"
    if state_path.exists():
        with open(state_path) as f:
            state = json.load(f)
        log_history = state.get("log_history", [])

if log_history:
    # Extract loss values
    train_steps = [e["step"] for e in log_history if "loss" in e]
    train_losses = [e["loss"] for e in log_history if "loss" in e]
    eval_steps = [e["step"] for e in log_history if "eval_loss" in e]
    eval_losses = [e["eval_loss"] for e in log_history if "eval_loss" in e]

    try:
        import matplotlib.pyplot as plt

        fig, ax = plt.subplots(1, 1, figsize=(10, 5))
        ax.plot(train_steps, train_losses, label="Train Loss", alpha=0.7)
        if eval_losses:
            ax.plot(eval_steps, eval_losses, label="Eval Loss", marker="o", markersize=4)
        ax.set_xlabel("Step")
        ax.set_ylabel("Loss")
        ax.set_title("LoRA Training Loss Curve")
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    except ImportError:
        print("matplotlib not available, displaying as text.")
        print("\nTraining loss trend:")
        for step, loss in zip(train_steps[-10:], train_losses[-10:]):
            bar = "█" * int(loss * 20)
            print(f"  Step {step:>6}: {loss:.4f} {bar}")

    # Summary metrics
    print("\n--- Training Metrics Summary ---")
    if train_losses:
        print(f"  Initial loss: {train_losses[0]:.4f}")
        print(f"  Final loss: {train_losses[-1]:.4f}")
        print(f"  Min loss: {min(train_losses):.4f}")
    if eval_losses:
        print(f"  Final eval loss: {eval_losses[-1]:.4f}")
        print(f"  Min eval loss: {min(eval_losses):.4f}")

    # GPU memory usage
    peak_vram = torch.cuda.max_memory_allocated() / (1024**3)
    print(f"\n  Peak VRAM usage: {peak_vram:.1f} GB")
    print(f"  Total training steps: {train_steps[-1] if train_steps else 'N/A'}")
else:
    print("Training logs not found.")

In [ ]:
"""Log training results to MLflow."""

from rhoai_model_training_lab.schemas.training import TrainingResult

# Build result object
lora_result = TrainingResult(
    method="lora",
    model_id=model_id,
    model_revision=model_revision,
    bundle_id=manifest.bundle_name,
    bundle_hash=manifest.bundle_hash,
    seed=lora_config["training"]["seed"],
    train_samples=manifest.canonical_train_count,
    validation_samples=manifest.canonical_validation_count,
    total_steps=train_steps[-1] if train_steps else 0,
    final_train_loss=train_losses[-1] if train_losses else 0.0,
    final_eval_loss=eval_losses[-1] if eval_losses else None,
    best_eval_loss=min(eval_losses) if eval_losses else None,
    wall_time_seconds=wall_time,
    peak_vram_gb=peak_vram,
    gpu_name=torch.cuda.get_device_name(0),
    checkpoint_path=output_dir,
)

# Log to MLflow
mlflow_uri = os.environ.get("MLFLOW_TRACKING_URI", "")
experiment_name = os.environ.get("MLFLOW_EXPERIMENT_TRAINING", "rhoai-model-training-lab-training")

if mlflow_uri:
    try:
        import mlflow

        mlflow.set_tracking_uri(mlflow_uri)
        mlflow.set_experiment(experiment_name)

        with mlflow.start_run(run_name=f"lora-{manifest.bundle_version}") as run:
            # Parameters
            mlflow.log_param("method", "lora")
            mlflow.log_param("model_id", model_id)
            mlflow.log_param("bundle_id", manifest.bundle_name)
            mlflow.log_param("bundle_version", manifest.bundle_version)
            mlflow.log_param("lora_r", lora_config["lora"]["r"])
            mlflow.log_param("lora_alpha", lora_config["lora"]["lora_alpha"])
            mlflow.log_param("learning_rate", lora_config["training_args"]["learning_rate"])
            mlflow.log_param("seed", lora_config["training"]["seed"])

            # Metrics
            mlflow.log_metric("final_train_loss", lora_result.final_train_loss)
            if lora_result.final_eval_loss is not None:
                mlflow.log_metric("final_eval_loss", lora_result.final_eval_loss)
            if lora_result.best_eval_loss is not None:
                mlflow.log_metric("best_eval_loss", lora_result.best_eval_loss)
            mlflow.log_metric("wall_time_seconds", lora_result.wall_time_seconds)
            mlflow.log_metric("peak_vram_gb", lora_result.peak_vram_gb)
            mlflow.log_metric("total_steps", lora_result.total_steps)
            mlflow.log_metric("train_samples", lora_result.train_samples)

            lora_result.mlflow_run_id = run.info.run_id
            lora_result.mlflow_experiment = experiment_name
            print(f"✅ MLflow logging complete (run_id: {run.info.run_id})")

    except Exception as exc:
        print(f"⚠️  MLflow logging failed: {exc}")
        print("  Training results have been saved locally.")
else:
    print("⚠️  MLFLOW_TRACKING_URI not set — saving results locally only.")

# Save result locally
result_path = Path(output_dir) / "training_result.json"
result_path.parent.mkdir(parents=True, exist_ok=True)
with open(result_path, "w") as f:
    f.write(lora_result.model_dump_json(indent=2))
print(f"📄 Training result saved: {result_path}")

In [ ]:
"""Verify checkpoint and proceed to export."""

from rich.table import Table
from rich.console import Console

console = Console()

# Verify checkpoint files exist
checkpoint_dir = Path(output_dir)
adapter_config = checkpoint_dir / "adapter_config.json"
adapter_model = checkpoint_dir / "adapter_model.safetensors"

# Find the best/latest checkpoint
checkpoints = sorted(checkpoint_dir.glob("checkpoint-*"), key=lambda p: p.name)
final_adapter = checkpoint_dir  # training_hub may save directly to output_dir

checks = []

# Check adapter config
has_config = adapter_config.exists() or any(
    (cp / "adapter_config.json").exists() for cp in checkpoints
)
checks.append(("Adapter config file", has_config))

# Check adapter weights
has_weights = adapter_model.exists() or any(
    list((cp).glob("adapter_model*")) for cp in checkpoints
)
checks.append(("Adapter weights file", has_weights))

# Check tokenizer
has_tokenizer = (checkpoint_dir / "tokenizer_config.json").exists() or any(
    (cp / "tokenizer_config.json").exists() for cp in checkpoints
)
checks.append(("Tokenizer files", has_tokenizer))

# Verify reload
reload_ok = False
try:
    from peft import PeftModel, PeftConfig

    # Try loading adapter config
    adapter_source = str(checkpoint_dir)
    if not adapter_config.exists() and checkpoints:
        adapter_source = str(checkpoints[-1])

    peft_config = PeftConfig.from_pretrained(adapter_source)
    reload_ok = True
    print(f"\n✅ Adapter config reload successful")
    print(f"   Base model: {peft_config.base_model_name_or_path}")
    print(f"   r={peft_config.r}, alpha={peft_config.lora_alpha}")
except Exception as exc:
    print(f"\n⚠️  Adapter reload verification failed: {exc}")

checks.append(("Adapter reload verification", reload_ok))

# Summary table
table = Table(title="🔍 LoRA Checkpoint Verification", show_header=True)
table.add_column("Item", style="bold")
table.add_column("Status")

for name, ok in checks:
    status = "✅" if ok else "❌"
    table.add_row(name, status)

console.print(table)

all_ok = all(ok for _, ok in checks)
if all_ok:
    print("\n🎉 LoRA training completed successfully!")
    print("\nNext steps:")
    print("  📓 04_osft_finetuning.ipynb — OSFT training (independent, same data)")
    print("  📓 05_export_and_deploy.ipynb — Model export and deployment")
else:
    print("\n⚠️  Some verification items failed. Check the results above.")